# Day 10 — v4 Training + Testing (RAG-Aware Multi-Model Pipeline)

**Jira Task KAN-54**: Day 10 [Retail + E-commerce + Manufacturing] — Running v4 Training + Testing (RAG-Aware)

### Workflow Overview:
1. **Mount Google Drive** & Install dependencies.
2. **Prepare v4 Datasets** (`train_v4.json`, `val_v4.json`, `test_v4.json`) with RAG context fields.
3. **Write v4 Configurations** for Qwen 2.5-7B and Llama-3-8B.
4. **Fine-Tune Qwen-RetailEcomManufacturing (v4)** via QLoRA.
5. **Fine-Tune Llama-RetailEcomManufacturing (v4)** via QLoRA.
6. **Index Knowledge Base** into ChromaDB Vector Database.
7. **Benchmark Both Models (100 queries each)** with RAG Retrieval + Generation.
8. **Plot Comparative Performance** (ROUGE-1, ROUGE-2, ROUGE-L, BLEU) and persist artifacts directly to Drive.

---  
## Step 1: Mount Google Drive & Install Required Packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q chromadb sentence-transformers transformers datasets accelerate peft bitsandbytes wandb trl rouge-score nltk pandas matplotlib huggingface_hub

---  
## Step 2: Initialize Workspace & Prepare v4 Datasets

In [ ]:
import os
import json
import shutil

project_dir = "/content/Retail"
gdrive_dir = None
for candidate in ["/content/drive/MyDrive/Retail LLM", "/content/drive/MyDrive/Retail"]:
    if os.path.isdir(candidate):
        gdrive_dir = candidate
        break
if gdrive_dir is None:
    gdrive_dir = "/content/drive/MyDrive/Retail LLM"
    print(f"[!] Defaulting Drive folder to: {gdrive_dir}")
else:
    print(f"[+] Active Drive folder detected: {gdrive_dir}")

for d in ["data/raw", "data/processed", "data/knowledge_base", "data/chroma_db", "configs", "src", "models", "models/evaluation"]:
    os.makedirs(os.path.join(project_dir, d), exist_ok=True)

# Copy existing splits from Drive if present
drive_processed = os.path.join(gdrive_dir, "data", "processed")
local_processed = os.path.join(project_dir, "data", "processed")
if os.path.isdir(drive_processed):
    print("[*] Copying data splits from Google Drive...")
    !cp -v "{drive_processed}/"*.json "{local_processed}/" 2>/dev/null || true

# Ensure v4 datasets are prepared
train_v4_path = os.path.join(local_processed, "train_v4.json")
val_v4_path = os.path.join(local_processed, "val_v4.json")
test_v4_path = os.path.join(local_processed, "test_v4.json")

if not os.path.exists(train_v4_path):
    print("[*] Generating train_v4.json, val_v4.json, and test_v4.json from available splits...")
    src_train = os.path.join(local_processed, "train_v3.json") if os.path.exists(os.path.join(local_processed, "train_v3.json")) else os.path.join(local_processed, "train.json")
    src_val = os.path.join(local_processed, "val_v3.json") if os.path.exists(os.path.join(local_processed, "val_v3.json")) else os.path.join(local_processed, "val.json")
    src_test = os.path.join(local_processed, "test_v3.json") if os.path.exists(os.path.join(local_processed, "test_v3.json")) else os.path.join(local_processed, "test.json")
    
    if os.path.exists(src_train):
        shutil.copyfile(src_train, train_v4_path)
    if os.path.exists(src_val):
        shutil.copyfile(src_val, val_v4_path)
    if os.path.exists(src_test):
        shutil.copyfile(src_test, test_v4_path)

for fname in ["train_v4.json", "val_v4.json", "test_v4.json"]:
    p = os.path.join(local_processed, fname)
    if os.path.exists(p):
        with open(p, "r") as f:
            cnt = len(json.load(f))
        print(f"[+] {fname}: {cnt} samples ({os.path.getsize(p)/(1024*1024):.2f} MB)")
    else:
        print(f"[-] Warning: {fname} missing")

---  
## Step 3: Write LoRA Configurations, Knowledge Base, and Source Scripts

In [ ]:
# 1. Write Configs
qwen_cfg = "{\n  \"model_type\": \"qwen\",\n  \"base_model_name_or_path\": \"Qwen/Qwen2.5-7B-Instruct\",\n  \"peft_config\": {\n    \"r\": 16,\n    \"lora_alpha\": 32,\n    \"lora_dropout\": 0.05,\n    \"bias\": \"none\",\n    \"task_type\": \"CAUSAL_LM\",\n    \"target_modules\": [\n      \"q_proj\",\n      \"k_proj\",\n      \"v_proj\",\n      \"o_proj\",\n      \"gate_proj\",\n      \"up_proj\",\n      \"down_proj\"\n    ]\n  },\n  \"quantization_config\": {\n    \"load_in_4bit\": true,\n    \"bnb_4bit_quant_type\": \"nf4\",\n    \"bnb_4bit_use_double_quant\": true,\n    \"bnb_4bit_compute_dtype\": \"float16\"\n  }\n}\n";
with open("/content/Retail/configs/qwen_lora_config_v4.json", "w", encoding="utf-8") as f:
    f.write(qwen_cfg)

llama_cfg = "{\n  \"model_type\": \"llama\",\n  \"base_model_name_or_path\": \"meta-llama/Meta-Llama-3-8B-Instruct\",\n  \"peft_config\": {\n    \"r\": 16,\n    \"lora_alpha\": 32,\n    \"lora_dropout\": 0.05,\n    \"bias\": \"none\",\n    \"task_type\": \"CAUSAL_LM\",\n    \"target_modules\": [\n      \"q_proj\",\n      \"k_proj\",\n      \"v_proj\",\n      \"o_proj\",\n      \"gate_proj\",\n      \"up_proj\",\n      \"down_proj\"\n    ]\n  },\n  \"quantization_config\": {\n    \"load_in_4bit\": true,\n    \"bnb_4bit_quant_type\": \"nf4\",\n    \"bnb_4bit_use_double_quant\": true,\n    \"bnb_4bit_compute_dtype\": \"float16\"\n  }\n}\n";
with open("/content/Retail/configs/llama_lora_config_v4.json", "w", encoding="utf-8") as f:
    f.write(llama_cfg)
print("[+] configs/qwen_lora_config_v4.json and configs/llama_lora_config_v4.json ready.")

# 2. Write Knowledge Base Manuals
retail_kb = "# Enterprise Retail & E-Commerce Customer Support Policies Manual\n\n## Section 1: Order Cancellation & Modification Policy\n- **Cancellation Window**: Customers can cancel orders within 60 minutes of placement directly from their account dashboard or by contacting customer support with their {{Order Number}}.\n- **Post-Dispatch Policy**: If an order has already entered the fulfillment or dispatched state, it cannot be canceled. The customer must wait for delivery and initiate a standard return.\n- **Address Changes**: Shipping address updates are permitted only while the order status is \"Processing\" and before label creation. Once tracking is generated, address modifications must be requested directly through the shipping courier (FedEx, UPS, DHL).\n\n## Section 2: Returns, Exchanges & Refunds\n- **Return Period**: We offer a 30-day return policy from the date of package delivery for all unworn, unused items with original tags and packaging intact.\n- **Return Initiation**: Customers must provide their {{Order Number}}, verify their {{Client Last Name}}, and select the return reason via the online portal to receive a prepaid return shipping label.\n- **Refund Processing Time**: Once the returned item is inspected at our fulfillment center, refunds are processed within 3 to 5 business days back to the original payment method (Credit Card, PayPal, or Bank Transfer). Store credit refunds are issued immediately upon return scanning.\n- **Damaged or Defective Items**: If an item arrives damaged or incorrect, customers must report the issue within 48 hours of delivery along with photographic evidence. Immediate free replacements or full refunds are provided without restocking fees.\n\n## Section 3: Accepted Payment Methods & Billing Inquiries\n- **Accepted Payment Gateways**: We accept Visa, MasterCard, American Express, Discover, PayPal, Apple Pay, Google Pay, and direct Wire/Bank Transfers.\n- **Payment Errors & Failed Transactions**: In the event of double charges or failed payment gateway authorizations, the pending authorization holds typically drop from the customer's bank statement within 24 to 72 hours.\n- **Invoice & Receipt Requests**: Detailed tax invoices with VAT/GST breakdowns are automatically sent to the registered email address and can also be downloaded from the \"Order History\" section.\n\n## Section 4: Shipping, Delivery & Tracking Guidelines\n- **Standard Shipping**: 3 to 5 business days delivery across all domestic zones.\n- **Express / Expedited Shipping**: 1 to 2 business days delivery with priority courier handling.\n- **Tracking Inquiries**: Customers can track package locations in real-time using their {{Tracking Number}} on our tracking portal or the carrier's official website.\n- **Delayed Shipments**: If tracking shows no movement for more than 4 business days, customer support will file a trace with the carrier and offer expedited reshipment or compensation credits.\n\n## Section 5: Account Security & Profile Management\n- **Password Reset & Account Recovery**: Customers can request a secure password reset link sent to their verified email. Multi-Factor Authentication (MFA) is recommended for all account tiers.\n- **Account Deletion / GDPR Requests**: Requests to permanently remove account data can be submitted through the Privacy Settings tab and are executed within 14 business days in compliance with data privacy regulations.\n";
with open("/content/Retail/data/knowledge_base/retail_ecommerce_policies.md", "w", encoding="utf-8") as f:
    f.write(retail_kb)

mfg_kb = "# Manufacturing Operations & Quality Control Standard Operating Procedures (SOP)\n\n## Section 1: Lean Six Sigma DMAIC Methodology\n- **Define Phase**: Explicitly identify the project scope, problem statement, customer CTQ (Critical to Quality) requirements, and form the cross-functional Project Charter.\n- **Measure Phase**: Establish baseline process capabilities, validate measurement systems using Gage R&R (Repeatability and Reproducibility < 10%), and collect accurate defect frequency data.\n- **Analyze Phase**: Identify root causes of variation or defect spikes using Cause-and-Effect (Fishbone/Ishikawa) diagrams, Pareto 80/20 analysis, and the 5 Whys interrogation technique.\n- **Improve Phase**: Design and implement targeted countermeasures, conduct Design of Experiments (DOE), streamline workflows with Kaizen events, and eliminate process bottlenecks.\n- **Control Phase**: Standardize the optimized process through updated SOPs, implement Poka-Yoke (mistake-proofing) mechanisms, and establish Statistical Process Control (SPC) monitoring plans.\n\n## Section 2: Statistical Process Control (SPC) & Control Charts\n- **X-bar and R Charts**: Used for continuous variable data collected in subgroups (e.g., component thickness, diameter, tensile strength). The X-bar chart tracks process average/central tendency, while the R chart tracks process dispersion/variation.\n- **P-Charts and C-Charts**: Used for attribute/discrete defect data. P-charts monitor the proportion of nonconforming units, while C-charts monitor the total count of defects per unit.\n- **Control Limits (UCL / LCL)**: Statistically calculated at +/- 3 standard deviations (+/- 3 Sigma) from the central line. Points falling beyond control limits or exhibiting non-random trends (e.g., 7 consecutive points on one side of the center line) indicate assignable causes that require immediate line pauses.\n\n## Section 3: Assembly Line Calibration & Maintenance Procedures\n- **Daily Calibration Protocol**: All optical measurement sensors, torque wrenches, and thermal bonding units must undergo zero-point calibration before shift commencement.\n- **Out-of-Specification Escalation**: If equipment drifts beyond +/- 0.05mm tolerance, the line operator must immediately tag the machine \"Out of Service\", notify the Shift Quality Lead, and quarantine all parts manufactured in the preceding 60-minute window.\n- **Preventative Maintenance (TPM)**: Total Productive Maintenance schedules dictate autonomous daily lubrication, weekly pneumatic pressure checks, and monthly sensor recalibration.\n\n## Section 4: Defect Root Cause Analysis (RCA) & Corrective Actions\n- **5 Whys Protocol**: For every critical non-conformance, trace the failure mode backward through at least 5 iterative \"Why\" queries to uncover the organizational or systemic failure rather than operator error.\n- **8D Problem Solving**: Used for major customer escapes or recurring defects. Follows the 8 disciplines from emergency response (D3) to permanent corrective actions (D5) and systemic prevention (D7).\n- **Corrective and Preventive Action (CAPA)**: Document all containment actions within 24 hours and establish root-cause validation within 5 business days.\n";
with open("/content/Retail/data/knowledge_base/manufacturing_sop_manual.md", "w", encoding="utf-8") as f:
    f.write(mfg_kb)
print("[+] Knowledge base manuals written.")

# 3. Write RAG Pipeline script
rag_pipe_code = "import os\nimport re\nimport glob\nimport torch\nimport chromadb\nfrom chromadb.utils import embedding_functions\n\nclass RetailMfgRAGPipeline:\n    def __init__(self, persist_dir=\"data/chroma_db\", collection_name=\"retail_mfg_knowledge\", embedding_model=\"all-MiniLM-L6-v2\"):\n        self.persist_dir = persist_dir\n        self.collection_name = collection_name\n        os.makedirs(self.persist_dir, exist_ok=True)\n        \n        print(f\"[*] Initializing ChromaDB Client at: {self.persist_dir}\")\n        self.client = chromadb.PersistentClient(path=self.persist_dir)\n        \n        print(f\"[*] Loading Embedding Model: {embedding_model}...\")\n        self.embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=embedding_model)\n        self.collection = self.client.get_or_create_collection(\n            name=self.collection_name,\n            embedding_function=self.embedding_fn,\n            metadata={\"hnsw:space\": \"cosine\"}\n        )\n        print(f\"[+] Collection '{self.collection_name}' ready. Records: {self.collection.count()}\")\n\n    def chunk_markdown_document(self, file_path):\n        chunks = []\n        with open(file_path, \"r\", encoding=\"utf-8\") as f:\n            content = f.read()\n        file_basename = os.path.basename(file_path)\n        domain = \"manufacturing\" if \"manufacturing\" in file_basename.lower() or \"sop\" in file_basename.lower() else \"retail\"\n        sections = re.split(r'\\n(?=##\\s+)', content)\n        for sec in sections:\n            sec = sec.strip()\n            if not sec:\n                continue\n            lines = sec.split(\"\\n\")\n            section_title = lines[0].replace(\"#\", \"\").strip()\n            body_paragraphs = [p.strip() for p in sec.split(\"\\n- \") if p.strip()]\n            for idx, para in enumerate(body_paragraphs):\n                if idx == 0 and \"\\n\" in para:\n                    para_parts = para.split(\"\\n\", 1)\n                    if len(para_parts) > 1 and para_parts[1].strip():\n                        para = para_parts[1].strip()\n                clean_text = para.replace(\"- \", \"\").replace(\"**\", \"\").strip()\n                if len(clean_text) > 30:\n                    prefix = \"Process Operations Manual: \" if domain == \"manufacturing\" else \"Customer Support Policy Guide: \"\n                    formatted_chunk = f\"{prefix}{section_title} - {clean_text}\"\n                    chunk_id = f\"{file_basename}_{section_title[:20]}_{idx}\".replace(\" \", \"_\").replace(\"/\", \"_\")\n                    chunks.append({\n                        \"id\": chunk_id,\n                        \"text\": formatted_chunk,\n                        \"metadata\": {\n                            \"source\": file_basename,\n                            \"section\": section_title,\n                            \"domain\": domain\n                        }\n                    })\n        return chunks\n\n    def index_knowledge_directory(self, kb_dir=\"data/knowledge_base\"):\n        doc_files = glob.glob(os.path.join(kb_dir, \"*.md\")) + glob.glob(os.path.join(kb_dir, \"*.txt\"))\n        if not doc_files:\n            return 0\n        all_chunks = []\n        for fpath in doc_files:\n            chunks = self.chunk_markdown_document(fpath)\n            all_chunks.extend(chunks)\n        if all_chunks:\n            self.collection.upsert(\n                ids=[c[\"id\"] for c in all_chunks],\n                documents=[c[\"text\"] for c in all_chunks],\n                metadatas=[c[\"metadata\"] for c in all_chunks]\n            )\n            print(f\"[+] Indexed {len(all_chunks)} passages into ChromaDB (Total: {self.collection.count()})\")\n        return len(all_chunks)\n\n    def retrieve_context(self, query, top_k=2):\n        results = self.collection.query(query_texts=[query], n_results=top_k)\n        retrieved_docs = results[\"documents\"][0] if results[\"documents\"] else []\n        retrieved_metas = results[\"metadatas\"][0] if results[\"metadatas\"] else []\n        return \" \".join(retrieved_docs), retrieved_docs, retrieved_metas\n\n    def generate_rag_response(self, model, tokenizer, query, top_k=2, max_new_tokens=150, temperature=0.7):\n        context_str, raw_docs, metadatas = self.retrieve_context(query, top_k=top_k)\n        prompt = (\n            f\"Below is an instruction that describes a task, paired with an input that provides further context. \"\n            f\"Write a response that appropriately completes the request.\\n\\n\"\n            f\"### Instruction:\\n{query}\\n\\n\"\n            f\"### Context:\\n{context_str}\\n\\n\"\n            f\"### Response:\\n\"\n        )\n        inputs = tokenizer(prompt, return_tensors=\"pt\").to(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n        prompt_len = inputs.input_ids.shape[1]\n        with torch.no_grad():\n            outputs = model.generate(\n                **inputs,\n                max_new_tokens=max_new_tokens,\n                temperature=temperature,\n                top_p=0.9,\n                do_sample=True,\n                pad_token_id=tokenizer.eos_token_id\n            )\n        generation_tokens = outputs[0][prompt_len:]\n        response = tokenizer.decode(generation_tokens, skip_special_tokens=True).strip()\n        return {\n            \"query\": query,\n            \"context\": context_str,\n            \"retrieved_passages\": raw_docs,\n            \"sources\": [m.get(\"source\", \"\") for m in metadatas],\n            \"response\": response\n        }\n";
with open("/content/Retail/src/rag_pipeline.py", "w", encoding="utf-8") as f:
    f.write(rag_pipe_code)

# 4. Write Training script
train_code = "import os\nimport argparse\nimport json\nimport torch\nfrom datasets import load_dataset\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\nfrom peft import LoraConfig, prepare_model_for_kbit_training\nfrom trl import SFTTrainer, SFTConfig\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description=\"QLoRA Fine-Tuning Pipeline for Retail & Manufacturing LLMs\")\n    parser.add_argument(\"--config\", type=str, required=True, help=\"Path to JSON config\")\n    parser.add_argument(\"--train_file\", type=str, default=\"data/processed/train_v4.json\", help=\"Train path\")\n    parser.add_argument(\"--val_file\", type=str, default=\"data/processed/val_v4.json\", help=\"Val path\")\n    parser.add_argument(\"--output_dir\", type=str, default=None, help=\"Output directory\")\n    parser.add_argument(\"--max_train_samples\", type=int, default=None)\n    parser.add_argument(\"--max_val_samples\", type=int, default=None)\n    return parser.parse_args()\n\ndef format_example(example):\n    instruction = example['instruction']\n    response = example['response']\n    context = example.get('context', '').strip()\n    if context:\n        example['text'] = (\n            f\"Below is an instruction that describes a task, paired with an input that provides further context. \"\n            f\"Write a response that appropriately completes the request.\\n\\n\"\n            f\"### Instruction:\\n{instruction}\\n\\n\"\n            f\"### Context:\\n{context}\\n\\n\"\n            f\"### Response:\\n{response}\"\n        )\n    else:\n        example['text'] = (\n            f\"Below is an instruction that describes a task. Write a response that appropriately completes the request.\\n\\n\"\n            f\"### Instruction:\\n{instruction}\\n\\n\"\n            f\"### Response:\\n{response}\"\n        )\n    return example\n\ndef main():\n    args = parse_args()\n    with open(args.config, \"r\", encoding=\"utf-8\") as f:\n        config = json.load(f)\n    model_type = config.get(\"model_type\", \"model\")\n    model_name = config.get(\"base_model_name_or_path\")\n    peft_settings = config.get(\"peft_config\", {})\n    quant_settings = config.get(\"quantization_config\", {})\n    if args.output_dir is None:\n        args.output_dir = f\"models/{model_type}_v4\"\n    print(f\"[+] Output Directory: {args.output_dir}\")\n\n    bnb_config = BitsAndBytesConfig(\n        load_in_4bit=quant_settings.get(\"load_in_4bit\", True),\n        bnb_4bit_quant_type=quant_settings.get(\"bnb_4bit_quant_type\", \"nf4\"),\n        bnb_4bit_use_double_quant=quant_settings.get(\"bnb_4bit_use_double_quant\", True),\n        bnb_4bit_compute_dtype=torch.float16\n    )\n\n    token = os.environ.get(\"HF_TOKEN\")\n    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, token=token)\n    tokenizer.padding_side = \"right\"\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n\n    model = AutoModelForCausalLM.from_pretrained(\n        model_name,\n        quantization_config=bnb_config,\n        device_map=\"auto\",\n        trust_remote_code=True,\n        torch_dtype=torch.float16,\n        token=token\n    )\n    for name, param in model.named_parameters():\n        if param.dtype == torch.bfloat16:\n            param.data = param.data.to(torch.float16)\n    for name, buf in model.named_buffers():\n        if buf.dtype == torch.bfloat16:\n            buf.data = buf.data.to(torch.float16)\n    model.config.torch_dtype = torch.float32\n    model = prepare_model_for_kbit_training(model)\n\n    lora_config = LoraConfig(\n        r=peft_settings.get(\"r\", 16),\n        lora_alpha=peft_settings.get(\"lora_alpha\", 32),\n        target_modules=peft_settings.get(\"target_modules\", []),\n        lora_dropout=peft_settings.get(\"lora_dropout\", 0.05),\n        bias=peft_settings.get(\"bias\", \"none\"),\n        task_type=\"CAUSAL_LM\"\n    )\n\n    dataset = load_dataset(\"json\", data_files={\"train\": args.train_file, \"validation\": args.val_file})\n    train_dataset = dataset[\"train\"]\n    val_dataset = dataset[\"validation\"]\n    if args.max_train_samples is not None:\n        train_dataset = train_dataset.select(range(min(len(train_dataset), args.max_train_samples)))\n    if args.max_val_samples is not None:\n        val_dataset = val_dataset.select(range(min(len(val_dataset), args.max_val_samples)))\n\n    train_dataset = train_dataset.map(format_example)\n    val_dataset = val_dataset.map(format_example)\n\n    training_args = SFTConfig(\n        output_dir=args.output_dir,\n        dataset_text_field=\"text\",\n        max_length=512,\n        num_train_epochs=3,\n        per_device_train_batch_size=4,\n        per_device_eval_batch_size=4,\n        gradient_accumulation_steps=4,\n        learning_rate=2e-4,\n        logging_steps=10,\n        eval_strategy=\"steps\",\n        eval_steps=50,\n        save_strategy=\"steps\",\n        save_steps=100,\n        save_total_limit=1,\n        fp16=True,\n        report_to=\"wandb\" if os.environ.get(\"WANDB_DISABLED\", \"\").lower() != \"true\" else \"none\",\n        lr_scheduler_type=\"cosine\",\n        remove_unused_columns=False\n    )\n    training_args.warmup_ratio = 0.03\n\n    trainer = SFTTrainer(\n        model=model,\n        train_dataset=train_dataset,\n        eval_dataset=val_dataset,\n        peft_config=lora_config,\n        processing_class=tokenizer,\n        args=training_args\n    )\n    for name, param in trainer.model.named_parameters():\n        if param.dtype == torch.bfloat16:\n            param.data = param.data.to(torch.float32)\n    for name, buf in trainer.model.named_buffers():\n        if buf.dtype == torch.bfloat16:\n            buf.data = buf.data.to(torch.float32)\n\n    resume_from_checkpoint = None\n    if args.output_dir and os.path.exists(args.output_dir):\n        checkpoints = [d for d in os.listdir(args.output_dir) if d.startswith(\"checkpoint-\") and os.path.isdir(os.path.join(args.output_dir, d))]\n        if checkpoints:\n            checkpoints.sort(key=lambda x: int(x.split(\"-\")[-1]))\n            resume_from_checkpoint = os.path.join(args.output_dir, checkpoints[-1])\n            print(f\"[+] Resuming from: {resume_from_checkpoint}\")\n\n    trainer.train(resume_from_checkpoint=resume_from_checkpoint)\n    trainer.model.save_pretrained(args.output_dir)\n    tokenizer.save_pretrained(args.output_dir)\n    print(f\"[+] Saved adapter weights to: {args.output_dir}\")\n\nif __name__ == \"__main__\":\n    main()\n";
with open("/content/Retail/src/train.py", "w", encoding="utf-8") as f:
    f.write(train_code)

# 5. Write Evaluate RAG script
eval_rag_code = "import os\nimport json\nimport random\nimport argparse\nimport torch\nimport nltk\nfrom nltk.translate.bleu_score import sentence_bleu, SmoothingFunction\nfrom rouge_score import rouge_scorer\nfrom transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig\nfrom peft import PeftModel\nfrom rag_pipeline import RetailMfgRAGPipeline\n\ntry:\n    nltk.download('punkt', quiet=True)\n    nltk.download('punkt_tab', quiet=True)\nexcept Exception:\n    pass\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description=\"Evaluate End-to-End RAG Pipeline on Test Queries\")\n    parser.add_argument(\"--model_id\", type=str, default=\"meta-llama/Meta-Llama-3-8B-Instruct\")\n    parser.add_argument(\"--adapter_dir\", type=str, required=True)\n    parser.add_argument(\"--test_file\", type=str, default=\"data/processed/test_v4.json\")\n    parser.add_argument(\"--kb_dir\", type=str, default=\"data/knowledge_base\")\n    parser.add_argument(\"--chroma_dir\", type=str, default=\"data/chroma_db\")\n    parser.add_argument(\"--output_file\", type=str, default=\"models/evaluation/rag_pipeline_v4_results.json\")\n    parser.add_argument(\"--num_samples\", type=int, default=100)\n    parser.add_argument(\"--seed\", type=int, default=42)\n    return parser.parse_args()\n\ndef main():\n    args = parse_args()\n    random.seed(args.seed)\n    print(f\"[*] Benchmarking {args.model_id} ({args.adapter_dir}) on {args.num_samples} queries...\")\n    rag_pipe = RetailMfgRAGPipeline(persist_dir=args.chroma_dir)\n    rag_pipe.index_knowledge_directory(args.kb_dir)\n    \n    token = os.environ.get(\"HF_TOKEN\")\n    if not token:\n        try:\n            from huggingface_hub import get_token\n            token = get_token()\n        except Exception:\n            token = None\n            \n    tokenizer = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True, token=token)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n        \n    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type=\"nf4\", bnb_4bit_compute_dtype=torch.float16)\n    base_model = AutoModelForCausalLM.from_pretrained(args.model_id, quantization_config=bnb_config, device_map=\"auto\", trust_remote_code=True, torch_dtype=torch.float16, token=token)\n    for name, param in base_model.named_parameters():\n        if param.dtype == torch.bfloat16:\n            param.data = param.data.to(torch.float16)\n    for name, buf in base_model.named_buffers():\n        if buf.dtype == torch.bfloat16:\n            buf.data = buf.data.to(torch.float16)\n            \n    adapter_path = args.adapter_dir\n    adapter_name = os.path.basename(adapter_path.rstrip(\"/\\\\"))\n    if \"{\" in adapter_path or not os.path.exists(adapter_path) or not os.path.exists(os.path.join(adapter_path, \"adapter_config.json\")):\n        candidates = [\n            f\"/content/drive/MyDrive/Retail LLM/models/{adapter_name}\",\n            f\"/content/drive/MyDrive/Retail/models/{adapter_name}\",\n            f\"/content/Retail/models/{adapter_name}\",\n            f\"models/{adapter_name}\"\n        ]\n        for candidate in candidates:\n            if os.path.exists(candidate) and os.path.exists(os.path.join(candidate, \"adapter_config.json\")):\n                adapter_path = candidate\n                break\n                \n    print(f\"[*] Attaching LoRA Adapter from: {adapter_path}...\")\n    model = PeftModel.from_pretrained(base_model, adapter_path)\n    model.eval()\n    \n    if not os.path.exists(args.test_file):\n        for fallback_name in [\"test_v4.json\", \"test_v3.json\", \"test.json\"]:\n            cand = os.path.join(os.path.dirname(args.test_file), fallback_name)\n            if os.path.exists(cand):\n                args.test_file = cand\n                break\n                \n    with open(args.test_file, \"r\", encoding=\"utf-8\") as f:\n        test_data = json.load(f)\n    eval_samples = random.sample(test_data, min(len(test_data), args.num_samples))\n    \n    rouge_scorer_inst = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)\n    smoothing = SmoothingFunction().method1\n    \n    results = []\n    total_r1, total_r2, total_rl, total_bleu = 0.0, 0.0, 0.0, 0.0\n    \n    for idx, sample in enumerate(eval_samples):\n        query = sample.get(\"instruction\", \"\")\n        reference = sample.get(\"response\", \"\")\n        rag_output = rag_pipe.generate_rag_response(model=model, tokenizer=tokenizer, query=query, top_k=2, max_new_tokens=150)\n        pred = rag_output[\"response\"]\n        \n        scores = rouge_scorer_inst.score(reference, pred)\n        r1, r2, rl = scores['rouge1'].fmeasure, scores['rouge2'].fmeasure, scores['rougeL'].fmeasure\n        ref_tokens = nltk.word_tokenize(reference.lower())\n        pred_tokens = nltk.word_tokenize(pred.lower())\n        bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothing)\n        \n        total_r1 += r1\n        total_r2 += r2\n        total_rl += rl\n        total_bleu += bleu\n        \n        results.append({\n            \"query\": query,\n            \"retrieved_context\": rag_output[\"context\"],\n            \"reference\": reference,\n            \"prediction\": pred,\n            \"sources\": rag_output[\"sources\"],\n            \"rouge1\": r1,\n            \"rouge2\": r2,\n            \"rougeL\": rl,\n            \"bleu\": bleu\n        })\n        if (idx + 1) % 10 == 0 or (idx + 1) == len(eval_samples):\n            print(f\"    [+] Processed {idx + 1}/{len(eval_samples)} | Current BLEU: {total_bleu/(idx+1):.4f}\", flush=True)\n            \n    n = len(eval_samples)\n    summary = {\n        \"model_id\": args.model_id,\n        \"adapter_dir\": adapter_path,\n        \"total_evaluated\": n,\n        \"mean_rouge1\": total_r1 / n,\n        \"mean_rouge2\": total_r2 / n,\n        \"mean_rougeL\": total_rl / n,\n        \"mean_bleu\": total_bleu / n\n    }\n    print(f\"[+] Final Scores: R1: {summary['mean_rouge1']:.4f} | R2: {summary['mean_rouge2']:.4f} | RL: {summary['mean_rougeL']:.4f} | BLEU: {summary['mean_bleu']:.4f}\", flush=True)\n    \n    os.makedirs(os.path.dirname(args.output_file), exist_ok=True)\n    with open(args.output_file, \"w\", encoding=\"utf-8\") as f:\n        json.dump({\"summary\": summary, \"results\": results}, f, indent=2, ensure_ascii=False)\n    print(f\"[+] Saved report to: {args.output_file}\")\n\nif __name__ == \"__main__\":\n    main()\n";
with open("/content/Retail/src/evaluate_rag.py", "w", encoding="utf-8") as f:
    f.write(eval_rag_code)
print("[+] All scripts and configs created successfully!")

---  
## Step 4: Login to Weights & Biases (W&B) and Hugging Face

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import wandb
import os

# HF Token resolution
token_candidates = ['HF_TOKEN', 'HF_TOKFI', 'HF_TOKEI', 'HF_token', 'hf_token', 'HF_TOKENS', 'TOKEN']
for secret_name in token_candidates:
    try:
        val = userdata.get(secret_name)
        if val:
            login(token=val, add_to_git_credential=False)
            os.environ['HF_TOKEN'] = val
            print(f"[+] Hugging Face token authenticated from secret: '{secret_name}'")
            break
    except Exception:
        pass

# W&B login
if os.environ.get('WANDB_DISABLED') != 'true':
    print("[*] Logging into Weights & Biases...")
    try:
        wandb.login()
    except Exception as e:
        print(f"[!] W&B login notice: {e}")
else:
    print("[+] W&B tracking disabled.")

---  
## Step 5: Fine-Tune Qwen-RetailEcomManufacturing (v4)
Training on `train_v4.json` (2,000 samples, 3 epochs) and saving checkpoints directly to Google Drive `models/qwen_v4`.

In [ ]:
qwen_v4_target = os.path.join(gdrive_dir, "models", "qwen_v4")
print(f"[*] Fine-Tuning Qwen v4 -> Target Output: {qwen_v4_target}")

!python -u /content/Retail/src/train.py \
    --config /content/Retail/configs/qwen_lora_config_v4.json \
    --train_file /content/Retail/data/processed/train_v4.json \
    --val_file /content/Retail/data/processed/val_v4.json \
    --output_dir "{qwen_v4_target}" \
    --max_train_samples 2000 \
    --max_val_samples 200

---  
## Step 6: Fine-Tune Llama-RetailEcomManufacturing (v4)
Training on `train_v4.json` (2,000 samples, 3 epochs) and saving checkpoints directly to Google Drive `models/llama_v4`.

In [ ]:
llama_v4_target = os.path.join(gdrive_dir, "models", "llama_v4")
print(f"[*] Fine-Tuning Llama v4 -> Target Output: {llama_v4_target}")

!python -u /content/Retail/src/train.py \
    --config /content/Retail/configs/llama_lora_config_v4.json \
    --train_file /content/Retail/data/processed/train_v4.json \
    --val_file /content/Retail/data/processed/val_v4.json \
    --output_dir "{llama_v4_target}" \
    --max_train_samples 2000 \
    --max_val_samples 200

---  
## Step 7: Index ChromaDB Vector Database

In [ ]:
import sys
sys.path.append('/content/Retail/src')
from rag_pipeline import RetailMfgRAGPipeline

pipeline = RetailMfgRAGPipeline(persist_dir="/content/Retail/data/chroma_db")
num_chunks = pipeline.index_knowledge_directory("/content/Retail/data/knowledge_base")
print(f"\n[+] ChromaDB vector database initialized with {num_chunks} enterprise knowledge chunks!")

---  
## Step 8: Benchmark Both Models on 100 RAG Test Queries

In [ ]:
print("\n=================== 1. BENCHMARKING QWEN v4 RAG ===================")
!python -u /content/Retail/src/evaluate_rag.py \
    --model_id Qwen/Qwen2.5-7B-Instruct \
    --adapter_dir "/content/drive/MyDrive/Retail LLM/models/qwen_v4" \
    --test_file /content/Retail/data/processed/test_v4.json \
    --output_file /content/Retail/models/evaluation/rag_qwen_v4_results.json \
    --num_samples 100

print("\n=================== 2. BENCHMARKING LLAMA v4 RAG ===================")
!python -u /content/Retail/src/evaluate_rag.py \
    --model_id meta-llama/Meta-Llama-3-8B-Instruct \
    --adapter_dir "/content/drive/MyDrive/Retail LLM/models/llama_v4" \
    --test_file /content/Retail/data/processed/test_v4.json \
    --output_file /content/Retail/models/evaluation/rag_llama_v4_results.json \
    --num_samples 100

---  
## Step 9: Comparative Analysis & Visualization (Qwen v4 vs Llama v4 RAG)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

eval_dir = "/content/Retail/models/evaluation"
qwen_res_file = os.path.join(eval_dir, "rag_qwen_v4_results.json")
llama_res_file = os.path.join(eval_dir, "rag_llama_v4_results.json")

results_summary = []

if os.path.exists(qwen_res_file):
    with open(qwen_res_file, "r") as f:
        data = json.load(f)
        s = data["summary"]
        s["Model"] = "Qwen-v4 + RAG"
        results_summary.append(s)

if os.path.exists(llama_res_file):
    with open(llama_res_file, "r") as f:
        data = json.load(f)
        s = data["summary"]
        s["Model"] = "Llama-v4 + RAG"
        results_summary.append(s)

if results_summary:
    df = pd.DataFrame(results_summary)
    disp_cols = ["Model", "total_evaluated", "mean_rouge1", "mean_rouge2", "mean_rougeL", "mean_bleu"]
    print("\n=================== DAY 10 RAG BENCHMARK COMPARISON ===================")
    display(df[disp_cols])

    # Side-by-side comparative bar chart
    metrics = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"]
    x = np.arange(len(metrics))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    for idx, row in df.iterrows():
        scores = [row["mean_rouge1"], row["mean_rouge2"], row["mean_rougeL"], row["mean_bleu"]]
        offset = (idx - 0.5) * width
        bars = ax.bar(x + offset, scores, width, label=row["Model"])
        for bar in bars:
            yval = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, yval + 0.01, f"{yval:.3f}", ha='center', fontsize=9, fontweight='bold')

    ax.set_ylabel("Metric Score (0 - 1.0)")
    ax.set_title("Day 10: Qwen-v4 vs Llama-v4 RAG Performance (100 Test Queries)")
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.set_ylim(0, 0.8)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("[!] Please run Step 8 first to generate benchmark reports.")

---  
## Step 10: Backup Models, ChromaDB Index & Evaluation Results to Google Drive

In [ ]:
print(f"[*] Persisting all Day 10 artifacts to Drive: {gdrive_dir}...")
drive_eval_dir = os.path.join(gdrive_dir, "models", "evaluation")
drive_chroma_dir = os.path.join(gdrive_dir, "data", "chroma_db")
os.makedirs(drive_eval_dir, exist_ok=True)
os.makedirs(drive_chroma_dir, exist_ok=True)

# Copy evaluation reports
!cp -v /content/Retail/models/evaluation/rag_*.json "{drive_eval_dir}/"

# Sync ChromaDB Vector Database
!rsync -av --progress /content/Retail/data/chroma_db/ "{drive_chroma_dir}/"
print("[+] All Day 10 v4 models, ChromaDB index, and benchmarks successfully persisted to Google Drive!")